# 05-aggregations

05-aggregations/01-aggregations.py
Aggregation patterns: basic aggs, countDistinct, groupBy, multi-column groupBy,
collect_list/set, conditional agg, HAVING equivalent, pivot, rollup, cube,
performance_reviews agg, first/last, approx_count_distinct.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), '..', '00-setup'))
from spark_session import get_spark, output_path, stop_and_wait
from seed_data import load_all, register_views
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = get_spark("05-aggregations")
dfs = register_views(spark)
emp  = dfs["employees"]
dept = dfs["departments"]
pr   = dfs["performance_reviews"]
po   = dfs["purchase_orders"]
sa   = dfs["salary_history"]


# ── Section 1: Basic aggregates ───────────────────────────────────────────────
# count("*") counts ALL rows including NULLs.
# count("salary") skips NULLs → 39 (emp 10 and 15 have NULL salary).
# avg/sum/stddev also skip NULL salary rows silently.
print("\n=== Section 1: Basic aggregates ===")
emp.agg(
    F.count("*").alias("total_rows"),            # 41
    F.count("salary").alias("non_null_salary"),  # 39 — emp 10, 15 excluded
    F.sum("salary").alias("total_salary"),
    F.avg("salary").alias("avg_salary"),
    F.min("salary").alias("min_salary"),         # 0.0 — emp 19 data flaw
    F.max("salary").alias("max_salary"),
    F.stddev("salary").alias("stddev_salary"),
).show()
# Observe: min_salary=0.0 (emp 19 soft-dup flaw), avg inflated by 350k CEO salary.


# ── Section 2: countDistinct ──────────────────────────────────────────────────
# countDistinct ignores NULLs — emp 22 NULL email is excluded from email count.
print("\n=== Section 2: countDistinct ===")
emp.agg(
    F.countDistinct("dept_id").alias("unique_depts"),
    F.countDistinct("job_title").alias("unique_titles"),
    F.countDistinct("status").alias("unique_statuses"),
).show()
# unique_statuses=2 (Active, Terminated — emp 35 is the only Terminated row)


# ── Section 3: groupBy + agg (per department) ─────────────────────────────────
# avg("salary") skips NULL salary rows within each group — dept 1 Engineering
# has emp 10 and 15 with NULL salary, so avg is computed over 12 non-NULL rows.
print("\n=== Section 3: groupBy + agg per department ===")
emp.groupBy("dept_id") \
   .agg(
       F.count("*").alias("emp_count"),
       F.avg("salary").alias("avg_salary"),
       F.sum("salary").alias("total_salary"),
       F.min("salary").alias("min_salary"),
       F.max("salary").alias("max_salary"),
   ) \
   .orderBy(F.col("emp_count").desc()) \
   .show()
# Engineering (dept_id=1) emp_count=14 — highest due to skew.
# dept_id=8 Executive has emp 41 (future hire_date) but no salary anomaly.


# ── Section 4: groupBy with multiple columns ──────────────────────────────────
# Cross-cuts department and employment status.
# emp 35 (Terminated, dept_id=5 HR) surfaces as a distinct group here.
print("\n=== Section 4: groupBy(dept_id, status) ===")
emp.groupBy("dept_id", "status") \
   .agg(
       F.count("*").alias("cnt"),
       F.avg("salary").alias("avg_salary"),
   ) \
   .orderBy("dept_id", "status") \
   .show()
# dept_id=5 will have two rows: Active and Terminated.


# ── Section 5: collect_list and collect_set ───────────────────────────────────
# collect_list preserves duplicates and ordering is non-deterministic.
# collect_set removes duplicates — useful for unique statuses per dept.
print("\n=== Section 5: collect_list and collect_set ===")
emp.groupBy("dept_id") \
   .agg(
       F.collect_list("last_name").alias("all_names"),
       F.collect_set("status").alias("unique_statuses"),
   ) \
   .orderBy("dept_id") \
   .show(truncate=False)
# dept_id=5 unique_statuses will contain both ['Active', 'Terminated'].


# ── Section 6: Conditional aggregation (CASE WHEN inside agg) ────────────────
# Sum of a boolean expression (1/0) counts rows matching a condition per group.
# This is the PySpark equivalent of SQL SUM(CASE WHEN ... THEN 1 ELSE 0 END).
print("\n=== Section 6: Conditional aggregation ===")
emp.groupBy("dept_id").agg(
    F.count("*").alias("total"),
    F.sum(F.when(F.col("status") == "Active",     1).otherwise(0)).alias("active_count"),
    F.sum(F.when(F.col("status") == "Terminated", 1).otherwise(0)).alias("terminated_count"),
    F.sum(F.when(F.col("salary").isNull(),         1).otherwise(0)).alias("null_salary_count"),
    F.sum(F.when(F.col("salary") == 0,             1).otherwise(0)).alias("zero_salary_count"),
).orderBy("dept_id").show()
# dept_id=1: null_salary_count=2 (emp 10, 15)
# dept_id=2: zero_salary_count=1 (emp 19, salary=0.0)
# dept_id=5: terminated_count=1 (emp 35)


# ── Section 7: HAVING equivalent (filter after groupBy) ──────────────────────
# In PySpark, filter() after groupBy().agg() acts as SQL HAVING.
# Cannot use WHERE on aggregated columns — must filter the aggregated result DF.
print("\n=== Section 7: HAVING equivalent — depts with more than 3 employees ===")
emp.groupBy("dept_id") \
   .agg(
       F.count("*").alias("cnt"),
       F.avg("salary").alias("avg_salary"),
   ) \
   .filter(F.col("cnt") > 3) \
   .orderBy(F.col("cnt").desc()) \
   .show()
# Only departments with 4+ employees appear: Engineering (14), Sales (7), etc.


# ── Section 8: pivot (employee count per dept per status) ────────────────────
# pivot() transposes row values into columns. Provide explicit value list to
# avoid a full scan to discover pivot values (better performance).
print("\n=== Section 8: pivot — emp count per dept per status ===")
emp.groupBy("dept_id") \
   .pivot("status", ["Active", "Terminated"]) \
   .count() \
   .orderBy("dept_id") \
   .show()
# dept_id=5: Active=1, Terminated=1
# All other depts: Active=N, Terminated=NULL (no Terminated rows in that dept)


# ── Section 9: rollup (hierarchical subtotals) ───────────────────────────────
# rollup(a, b) produces: (a, b), (a, NULL) subtotal, (NULL, NULL) grand total.
# NULL in grouping columns indicates a subtotal or grand total row — NOT a data NULL.
print("\n=== Section 9: rollup(dept_id, status) — hierarchical subtotals ===")
emp.rollup("dept_id", "status") \
   .agg(
       F.count("*").alias("cnt"),
       F.sum("salary").alias("total_salary"),
   ) \
   .orderBy(F.col("dept_id").asc_nulls_last(), F.col("status").asc_nulls_last()) \
   .show(30)
# Row where dept_id=NULL, status=NULL → grand total across all employees.
# Row where dept_id=X, status=NULL → subtotal for that department across all statuses.


# ── Section 10: cube (all combinations of grouping columns) ───────────────────
# cube(a, b) produces all 2^2 combinations: (a,b), (a,NULL), (NULL,b), (NULL,NULL).
# More combinations than rollup — useful for cross-dimensional reporting.
print("\n=== Section 10: cube(dept_id, status) — all combinations ===")
emp.cube("dept_id", "status") \
   .agg(F.count("*").alias("cnt")) \
   .orderBy(F.col("dept_id").asc_nulls_last(), F.col("status").asc_nulls_last()) \
   .show(20)
# Extra rows vs rollup: (NULL, 'Active'), (NULL, 'Terminated') — cross-dept totals per status.


# ── Section 11: agg on performance_reviews (avg rating per period) ─────────────
# rating is nullable — two reviews have NULL rating (review 5 and 9).
# count("rating") excludes NULLs; avg("rating") also excludes NULLs automatically.
print("\n=== Section 11: performance_reviews — avg rating per review period ===")
pr.groupBy("review_period") \
  .agg(
      F.count("*").alias("total_reviews"),
      F.count("rating").alias("with_rating"),   # NULLs excluded
      F.avg("rating").alias("avg_rating"),
      F.min("rating").alias("min_rating"),
      F.max("rating").alias("max_rating"),
  ) \
  .orderBy("review_period") \
  .show()
# 2024-H1 has 1 review (review 5) with NULL rating → with_rating=0, avg_rating=NULL.


# ── Section 12: first and last (non-deterministic without ORDER BY) ─────────────
# Without a Window spec, first() and last() return arbitrary values from the group.
# Use with Window.orderBy() for deterministic results (see 06-window-functions).
print("\n=== Section 12: first and last per employee in salary_history ===")
sa.groupBy("emp_id") \
  .agg(
      F.first("salary_after").alias("first_seen_salary"),   # non-deterministic
      F.last("salary_after").alias("last_seen_salary"),     # non-deterministic
      F.count("*").alias("change_count"),
  ) \
  .orderBy("emp_id") \
  .show(5)
# emp_id=2 has change_count=3; emp_id=5 has change_count=4 (includes duplicate row 16).
# Use Window + row_number() for deterministic first/last by effective_date.


# ── Section 13: approx_count_distinct (HyperLogLog) ───────────────────────────
# approx_count_distinct uses HyperLogLog sketch — O(1) memory, trades accuracy for speed.
# rsd=0.05 means ~5% relative standard deviation. Useful on large datasets.
# On this small dataset (41 rows) the approximation may exactly match exact count.
print("\n=== Section 13: approx_count_distinct vs countDistinct ===")
emp.agg(
    F.approx_count_distinct("job_title", rsd=0.05).alias("approx_unique_titles"),
    F.countDistinct("job_title").alias("exact_unique_titles"),
).show()
# Both should be close (or equal) here since data is tiny.
# At scale (millions of rows), approx_count_distinct is significantly faster.


# ── Keep session alive for Spark UI inspection ────────────────────────────────
stop_and_wait(spark)